# Imports & Functions

In [1]:
import pandas as pd
import numpy as np
from functools import partial
import optuna.visualization as vis
import plotly.express as px

from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sklearn.ensemble import HistGradientBoostingRegressor
import xgboost as xgb
import optuna

In [2]:
def remove_top_1_percent_outliers(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    df_clean = df.copy()
    for col in features:
        lower_bound = np.percentile(df_clean[col], 1)
        upper_bound = np.percentile(df_clean[col], 99)
        
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    
    return df_clean

def remove_outliers(df, target, threshold=0.012):
    """Removes the extreme outliers"""
    lower_bound = target.quantile(threshold)
    upper_bound = target.quantile(1 - threshold)
    
    mask = (target >= lower_bound) & (target <= upper_bound)
    return df[mask], target[mask]

def frequency_encoding(df_to_modify: pd.DataFrame, df_initial: pd.DataFrame, column: str) -> pd.DataFrame:
    freq_map = df_initial[column].value_counts()
    df_to_modify[column + "_freq"] = df_initial[column].map(freq_map).fillna(0)
    return df_to_modify

# Function to optimize the parameters of the XGBoost
def objective(trial, X_train_clean, Y_train_clean, X_val, y_val):
    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
        "objective": "reg:squarederror",
        "eval_metric": "mae"
    }

    dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
    dval = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    preds = model.predict(dval)
    preds = preds.round().astype(int)
    mae = mean_absolute_error(y_val, preds)
    return mae


# Read files & plot basic information

In [23]:
x_test_file = pd.read_csv(r"x_test_final.csv")
x_train_file = pd.read_csv(r"x_train_final.csv")
y_sample = pd.read_csv(r"y_sample_final.csv")
y_train_file = pd.read_csv(r"y_train_final_j5KGWWK.csv")

In [24]:
# Get the frequency of each unique value of p0q0
target_counts = y_train_file["p0q0"].value_counts(normalize=True).sort_index()
target_df = pd.DataFrame({"p0q0": target_counts.index, "count": target_counts.values})

fig = px.bar(target_df, x="p0q0", y="count", title="Delay Frequency in Y train file",
             labels={"p0q0": "Delay (in minutes)", "count": "Frequency"})
fig.update_layout(xaxis=dict(dtick=10))  # pour forcer les pas de 1 sur l'axe X
fig.show()

# Data & Engineering

### Train Data

In [25]:
###### Train through Sklearn #####

x_train, x_val, y_train, y_val = train_test_split(x_train_file, y_train_file, test_size=0.1, random_state=54)
y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

In [6]:
##### Train through simple way #####

split_idx = int(len(x_train_file) * 0.9)
x_train, x_val = x_train_file.iloc[:split_idx], x_train_file.iloc[split_idx:]
y_train, y_val = y_train_file.iloc[:split_idx], y_train_file.iloc[split_idx:]

y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

### Outliers & Encoding

In [7]:
###################################
##### OLD VERSION OF OUTLIERS #####
###################################

outlier_features = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
x_train = remove_top_1_percent_outliers(x_train, outlier_features)
y_train = y_train.loc[x_train.index]

In [26]:
##### Encoding #####

features_to_keep = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
X_train = x_train[features_to_keep].copy()
X_val = x_val[features_to_keep].copy()
X_test = x_test_file[features_to_keep].copy()
Y_train = y_train.copy()

# Feature Engineering: Encode "gare" and "arret" column
gare_counts = x_train['gare'].value_counts()
X_train['gare_encoded'] = x_train['gare'].map(gare_counts)
X_val['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)
X_test['gare_encoded'] = x_test_file['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X_train['arret_encoded'] = x_train['arret'].map(arret_counts)
X_val['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)
X_test['arret_encoded'] = x_test_file['arret'].map(arret_counts).fillna(0)

In [27]:
##### Remove Outliers #####

X_train_clean, Y_train_clean = remove_outliers(X_train, Y_train, threshold=0.012)

# Ensure validation set has same columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Weighted Average Model

In [10]:
## First method with weighted mean

x_test_final_test = x_test_file.copy()
x_test_final_test["p0q0"] = (0.6 * x_test_final_test["p0q2"] + 0.3 * x_test_final_test["p0q3"] + 0.1 * x_test_final_test["p0q4"]).round(0)
y_first_test = x_test_final_test["p0q0"]
y_first_test.to_csv("Weighted_Average.csv")

# Random Forest Model

In [ ]:
# Initialise the model
rf = RandomForestRegressor(n_estimators=100, random_state=54)

# Train the model
rf.fit(X_train, Y_train.values.ravel())

# Predecit on all values
y_pred = rf.predict(X_val)

# Evaluate the model
mae = mean_absolute_error(y_val, y_pred)
print(f"Mean Absolute Error (MAE) : {mae}")


Erreur absolue moyenne (MAE) : 0.7934804381077231


In [29]:
y_test_pred = rf.predict(X_test)
submission = pd.DataFrame({'p0q0': y_test_pred})
submission.to_csv("submission_RF.csv", index=False)

# HGB Model

In [ ]:
# Boosted gradient model
hgb = HistGradientBoostingRegressor(
    max_iter=1000,
    max_depth=50,
    learning_rate=0.2,
    max_bins=255,
    l2_regularization=1,
    random_state=42)

hgb.fit(X_train, Y_train)

y_pred = hgb.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
print(f"Mean Absolute Error (MAE) : {mae}")

Erreur absolue moyenne (MAE) : 0.7691055674285049


In [31]:
# Predict the test file
y_test_pred = hgb.predict(X_test)

# Create the good file
submission = pd.DataFrame({
    "Unnamed: 0": y_sample["Unnamed: 0"],
    "p0q0": y_test_pred
})

submission.to_csv("submission_hgb.csv", index=False)

# XGBoost Model

In [ ]:
# Initialize the model
xgb_model = xgb.XGBRegressor(
    n_estimators=1400,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.009,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    reg_alpha=1,      # L1 (lasso)
    reg_lambda=2,     # L2 (ridge)
    random_state=54,
    n_jobs=-1           # Use all CPU cores
)

# Fit the model
xgb_model.fit(
    X_train_clean,
    Y_train_clean,
)

# Predict
y_pred = xgb_model.predict(X_val)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)

# Evaluate
mae = mean_absolute_error(y_val, y_pred)
print(f"Mean Absolute Error (MAE) : {mae}")

Erreur absolue moyenne (MAE) avec XGBoost : 0.6360094114826083


In [ ]:
# Predict on test
y_test_pred = xgb_model.predict(X_test)
y_test_pred = y_test_pred.round(0).astype(int)
y_test_pred = pd.DataFrame(y_test_pred)

# Save in a csv file
y_test_pred.to_csv("submission_xgboost9.csv")

print("Submission file saved : submission_xgboost9.csv")


Fichier de soumission sauvegardé : submission_xgboost9.csv


# XGBoost Model with optimized parameters

### Search the best parameters an plot

In [ ]:
objective_with_data = partial(
    objective,
    X_train_clean=X_train_clean,
    Y_train_clean=Y_train_clean,
    X_val=X_val,
    y_val=y_val
)

study = optuna.create_study(direction="minimize")
study.optimize(objective_with_data, n_trials=30)

print("Best Hyperparameters :")
print(study.best_params)

[I 2025-03-31 15:47:18,282] A new study created in memory with name: no-name-ce46682b-8cd3-46ee-a59f-026a6d3c77bc
[I 2025-03-31 15:47:30,412] Trial 0 finished with value: 0.6335216628950799 and parameters: {'max_depth': 11, 'learning_rate': 0.029964353148352038, 'subsample': 0.7848350665989051, 'colsample_bytree': 0.9329064954268251, 'reg_alpha': 1.4881590658274113, 'reg_lambda': 4.309794541103191}. Best is trial 0 with value: 0.6335216628950799.
[I 2025-03-31 15:47:37,915] Trial 1 finished with value: 0.6356197641134773 and parameters: {'max_depth': 13, 'learning_rate': 0.04013597404189441, 'subsample': 0.6363506993320587, 'colsample_bytree': 0.942633125727001, 'reg_alpha': 2.3096219871010666, 'reg_lambda': 1.426468529256721}. Best is trial 0 with value: 0.6335216628950799.
[I 2025-03-31 15:47:45,243] Trial 2 finished with value: 0.6339113102642109 and parameters: {'max_depth': 12, 'learning_rate': 0.04393672871357175, 'subsample': 0.9497732541953836, 'colsample_bytree': 0.93453875527

🎯 Meilleurs hyperparamètres trouvés :
{'max_depth': 14, 'learning_rate': 0.014791428667443717, 'subsample': 0.973773462490931, 'colsample_bytree': 0.7405247566834626, 'reg_alpha': 4.9691285687391975, 'reg_lambda': 0.5321492768705345}


In [34]:
## Plot the results of the parameters optimization

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

### Train the model

In [ ]:
best_params = study.best_params
best_params.update({
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "n_jobs": -1,
    "random_state": 42
})

dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
dval = xgb.DMatrix(X_val, label=y_val)

best_xgboost_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, "validation")],
    early_stopping_rounds=50,
    verbose_eval=True,
)

y_pred = best_xgboost_model.predict(dval).round().astype(int)
mae = mean_absolute_error(y_val, y_pred)
print(f"MAE with best parameters : {mae:.4f}")


[0]	validation-mae:0.90366
[1]	validation-mae:0.89859
[2]	validation-mae:0.89468
[3]	validation-mae:0.89067
[4]	validation-mae:0.88655
[5]	validation-mae:0.88368
[6]	validation-mae:0.88078
[7]	validation-mae:0.87822
[8]	validation-mae:0.87406
[9]	validation-mae:0.87178
[10]	validation-mae:0.86911
[11]	validation-mae:0.86536
[12]	validation-mae:0.86200
[13]	validation-mae:0.85987
[14]	validation-mae:0.85689
[15]	validation-mae:0.85389
[16]	validation-mae:0.85193
[17]	validation-mae:0.85016
[18]	validation-mae:0.84864
[19]	validation-mae:0.84560
[20]	validation-mae:0.84322
[21]	validation-mae:0.84165
[22]	validation-mae:0.83905
[23]	validation-mae:0.83743
[24]	validation-mae:0.83564
[25]	validation-mae:0.83313
[26]	validation-mae:0.83110
[27]	validation-mae:0.82982
[28]	validation-mae:0.82748
[29]	validation-mae:0.82514
[30]	validation-mae:0.82371
[31]	validation-mae:0.82231
[32]	validation-mae:0.82097
[33]	validation-mae:0.81877
[34]	validation-mae:0.81699
[35]	validation-mae:0.81511
[3

### Error plots

In [36]:
## Print the distribution of the error ##

errors = y_val - y_pred
fig = px.histogram(errors, nbins=100, title="Error Distribution (y_val - y_pred)")
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(xaxis_title="Error", yaxis_title="Number of occurences")
fig.show()

## Print the distribution of the error in absolute value ##

abs_errors = np.abs(y_val - y_pred)
fig = px.histogram(abs_errors, nbins=100, title="Absolute Error Distribution (y_val - y_pred)")
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(xaxis_title="Absolute Error", yaxis_title="Number of occurences")
fig.show()

### Variables importance

In [ ]:
importance = best_xgboost_model.get_score(importance_type='gain')
importance_df = pd.DataFrame({
    "feature": list(importance.keys()),
    "importance": list(importance.values())
}).sort_values(by="importance", ascending=False).head(10)

fig = px.bar(importance_df, x="importance", y="feature", orientation="h",
             title="Top 10 most important variables (gain)")
fig.update_layout(yaxis_title="Variables", xaxis_title="Gain")
fig.show()

### Create the submission file

In [ ]:
X_full = pd.concat([X_train_clean, X_val], axis=0)
Y_full = pd.concat([Y_train_clean, y_val], axis=0)
dtrain_full = xgb.DMatrix(X_full, label=Y_full)
X_test = X_test.reindex(columns=X_train_clean.columns, fill_value=0)
dtest = xgb.DMatrix(X_test)

final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=best_xgboost_model.best_iteration + 50
)

# Predictions
y_test_pred = final_model.predict(dtest)
y_test_pred = y_test_pred.round(0).astype(int)

y_test_pred = pd.DataFrame(y_test_pred, columns=["p0q0"])
y_test_pred.index.name = "index"

y_test_pred.to_csv("submission_xgboost12.csv")

print("Submission file saved : submission_xgboost12.csv")


Fichier de soumission sauvegardé : submission_xgboost12.csv
